In [21]:
import os
from cdo import Cdo

In [22]:
cdo = Cdo(cdo='/usr/local/apps/cdo/2.4.0/bin/cdo')

# Helper functions

In [23]:
def annual_mean(input_folder, output_folder, cdo,
                filename_filter=None, exclude_filter=None):
    """
    Compute annual means from monthly NetCDF files using CDO yearmean.
    """

    os.makedirs(output_folder, exist_ok=True)

    for filename in os.listdir(input_folder):

        if not filename.endswith(".nc"):
            continue

        if filename_filter and filename_filter not in filename:
            continue

        if exclude_filter and exclude_filter in filename:
            continue

        input_file = os.path.join(input_folder, filename)
        output_file = os.path.join(output_folder, f"yearmean_{filename}")

        if not os.path.exists(output_file):

            cdo.yearmean(
                input=input_file,
                output=output_file
            )

            print(f"✅ Annual mean: {filename}")

        else:
            print(f"⏩ Skipped: {filename}")

In [24]:
def remap_files(input_folder, output_folder, file_filter, method="remapbil"):
    """
    Remap netCDF files to a target grid.

    Parameters
    ----------
    input_folder : str
        Folder containing input NetCDF files.
    output_folder : str
        Folder for remapped files.
    file_filter : str
        String that must appear in the filename.
    method : str
        CDO remapping operator: "remapcon", "remapbil", "remapnn", etc.
    """

    os.makedirs(output_folder, exist_ok=True)

    for filename in os.listdir(input_folder):
        if filename.endswith(".nc") and file_filter in filename:

            input_file = os.path.join(input_folder, filename)
            output_file = os.path.join(output_folder, f"remapped_{filename}")

            if not os.path.exists(output_file):

                if method == "remapcon":
                    cdo.remapcon(
                        "r180x90",
                        input=input_file,
                        output=output_file
                    )

                elif method == "remapbil":
                    cdo.remapbil(
                        "r180x90",
                        input=input_file,
                        output=output_file
                    )

                elif method == "remapnn":
                    cdo.remapnn(
                        "r180x90",
                        input=input_file,
                        output=output_file
                    )

                else:
                    raise ValueError(f"Unknown remapping method: {method}")

                print(f"✅ Remapped: {filename}")

            else:
                print(f"⏩ Skipped (already exists): {filename}")

In [25]:
def extract_variables(cdo, input_folder, output_folder, variables,
                      filename_filter=None, exclude_filter=None,
                      year_range=None):
    """
    Extract selected variables from remapped files.
    Optional year filtering supported.
    """
    os.makedirs(output_folder, exist_ok=True)

    for filename in os.listdir(input_folder):
        if not filename.endswith(".nc"):
            continue

        if filename_filter and filename_filter not in filename:
            continue

        if exclude_filter and exclude_filter in filename:
            continue

        # Optional year filter
        if year_range:
            try:
                year_str = filename.split("_")[-1].replace(".nc", "")
                year = int(year_str.split("-")[0])
                if not (year_range[0] <= year <= year_range[1]):
                    continue
            except:
                continue

        infile = os.path.join(input_folder, filename)
        outfile = os.path.join(output_folder, f"small_{filename}")

        if not os.path.exists(outfile):
            cdo.selname(variables, input=infile, output=outfile)
            print(f"✅ Extracted vars from {filename}")
        else:
            print(f"⏩ Skipped (already exists): {filename}")

# Experiments configuration

In [12]:
experiments = {
    #"exp3": "/lus/h2resw01/scratch/itcv/ece4/exp3/output/",
    #"XPPI": "/lus/h2resw01/scratch/ccpd/ece4/XPPI/output/",
    #"XEPI": "/lus/h2resw01/scratch/ccpd/ece4/XEPI/output/",
    "XE3C": "/lus/h2resw01/scratch/ccpd/ece4/XE3C/output/",
    #"XE6C": "/lus/h2resw01/scratch/ccpd/ece4/XE6C/output/",
}

In [26]:
experiments = {"PV19": "/lus/h2resw01/scratch/ecme3497/ece4/PV19/output/"}

In [27]:
base_output = "/lus/h2resw01/hpcperm/ecme3497/data-analysis/epochal/"

atm_vars = "tas,pr,rsut,rlut,rsdt"
oce_vars = "tos,sos"

# Main processing loop

In [32]:
for exp_name, input_base in experiments.items():

    print(f"\n================ {exp_name} ================\n")

    # -------------------------
    # Extract ATM variables
    # -------------------------
    extract_variables(
        input_folder=os.path.join(input_base, "oifs"),
        output_folder=os.path.join(base_output, exp_name, "variables/atm"),
        variables=atm_vars,
        filename_filter="atm_cmip6_1m",
        exclude_filter="_pl_",
        cdo = cdo
    )

    # -------------------------
    # Extract OCE variables
    # -------------------------
    extract_variables(
        input_folder=os.path.join(input_base, "nemo"),
        output_folder=os.path.join(base_output, exp_name, "variables/oce"),
        variables=oce_vars,
        filename_filter="oce_1m_T",
        cdo=cdo
    )

    # ------------------------
    # Compute annual means
    # ------------------------
    annual_mean(
        input_folder=os.path.join(base_output, exp_name, "variables/atm"),
        output_folder=os.path.join(base_output, exp_name, "annual_mean/atm"),
        cdo=cdo
    )

    annual_mean(
        input_folder=os.path.join(base_output, exp_name, "variables/oce"),
        output_folder=os.path.join(base_output, exp_name, "annual_mean/oce"),
        cdo=cdo
    )

    # -------------------------
    # Remap atmosphere (OIFS)
    # -------------------------
    remap_files(
        input_folder=os.path.join(base_output, exp_name, "annual_mean/atm"),
        output_folder=os.path.join(base_output, exp_name, "remaped/atm"),
        file_filter="_1m_",
        method="remapcon"
    )

    # -------------------------
    # Remap ocean (NEMO)
    # -------------------------
    remap_files(
        input_folder=os.path.join(base_output, exp_name, "annual_mean/oce"),
        output_folder=os.path.join(base_output, exp_name, "remaped/oce"),
        file_filter="oce_1m_T",
        method="remapbil"
    )




================ PV19 ================

⏩ Skipped (already exists): PV19_atm_cmip6_1m_1995-1995.nc
⏩ Skipped (already exists): PV19_atm_cmip6_1m_1993-1993.nc
⏩ Skipped (already exists): PV19_atm_cmip6_1m_1992-1992.nc
⏩ Skipped (already exists): PV19_atm_cmip6_1m_1991-1991.nc
⏩ Skipped (already exists): PV19_atm_cmip6_1m_1990-1990.nc
⏩ Skipped (already exists): PV19_atm_cmip6_1m_1994-1994.nc
⏩ Skipped (already exists): PV19_oce_1m_T_1995-1995.nc
⏩ Skipped (already exists): PV19_oce_1m_T_1991-1991.nc
⏩ Skipped (already exists): PV19_oce_1m_T_1994-1994.nc
⏩ Skipped (already exists): PV19_oce_1m_T_1990-1990.nc
⏩ Skipped (already exists): PV19_oce_1m_T_1993-1993.nc
⏩ Skipped (already exists): PV19_oce_1m_T_1992-1992.nc
⏩ Skipped: small_PV19_atm_cmip6_1m_1993-1993.nc
⏩ Skipped: small_PV19_atm_cmip6_1m_1994-1994.nc
⏩ Skipped: small_PV19_atm_cmip6_1m_1992-1992.nc
⏩ Skipped: small_PV19_atm_cmip6_1m_1990-1990.nc
⏩ Skipped: small_PV19_atm_cmip6_1m_1995-1995.nc
⏩ Skipped: small_PV19_atm_cmip6_1m_